<div style="
    width: 100%;
    box-sizing: border-box;
    margin: 20px 0;
    padding: 20px;
    background: #0b0f1a;
    border: 1px solid rgba(0,255,255,0.2);
    border-radius: 12px;
    color: #d6faff;
    font-family: Arial, sans-serif;
">

<h2 style="color:#00f5ff;">⚡ Professionally You — LLM Tool Use & Deployment Exercise</h2>

<p>
Build a professional AI career chatbot representing a specific persona (<span style="color:#00f5ff;">Shakiru Sikiru</span>) by grounding the system prompt with context extracted from a <span style="color:#39ff14;">LinkedIn PDF profile</span> and a <span style="color:#39ff14;">summary.txt</span> file. The agent must respond on your behalf to career-related queries in a professional, factual tone — acting as your live digital resume.
</p>

<p>
Integrate <strong>Tool Use</strong> via two callable functions the LLM can invoke autonomously:
<span style="color:#00f5ff;">record_user_details</span> (to capture a recruiter's contact info) and
<span style="color:#00f5ff;">record_unknown_question</span> (to log any question the agent couldn't answer).
Both tools dispatch a <strong>real-time Pushover push notification</strong> to your phone whenever they are triggered.
</p>

<p>
Implement the full <strong>tool-call loop</strong> — if the model returns <code>finish_reason == "tool_calls"</code>, the agent must execute the requested tool, append the result back into the message history, and automatically re-invoke the LLM for a final response, completing the agentic cycle.
</p>

<h2 style="color:#ff7800; margin-top: 20px;">🚀 Deployment — HuggingFace Spaces</h2>

<p>Deploy the Gradio chat interface publicly to <span style="color:#39ff14;">HuggingFace Spaces</span> using the following steps:</p>

<ol style="line-height: 2;">
  <li>Create a HuggingFace account at <span style="color:#00f5ff;">https://huggingface.co</span> and generate a <strong>WRITE-permission Access Token</strong>.</li>
  <li>Install the HuggingFace CLI: <code>uv tool install 'huggingface_hub[cli]'</code> then authenticate: <code>hf auth login --token hf_xxx</code>.</li>
  <li>Add <code>HF_TOKEN=hf_xxx</code> to your <code>.env</code> file.</li>
  <li>From the <code>1_foundations</code> folder, run: <code>uv run gradio deploy</code> — name it <span style="color:#00f5ff;">career_conversation</span>, select <strong>cpu-basic</strong> hardware, and supply your API keys and Pushover secrets when prompted.</li>
  <li>Update the <code>me/</code> directory with your own LinkedIn PDF and <code>summary.txt</code>, and change <code>self.name</code> in <code>app.py</code> to your own name before deploying.</li>
</ol>


</div>

In [1]:
# imports relevant libraries

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [2]:
# Load the API keys into environment variables

load_dotenv(override=True)

True

In [3]:
# Get the API keys from environment variables

openai_api_key = os.getenv('NVIDIA_API_KEY_1')
google_api_key = os.getenv('GOOGLE_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')
deepseek_api_key = os.getenv('NVIDIA_API_KEY_2')
groq_api_key = os.getenv('GROQ_API_KEY')

In [4]:
# Initialize the OpenAI client with the API key and base URL

openai = OpenAI(
    base_url = "https://integrate.api.nvidia.com/v1", 
    api_key = openai_api_key
)

In [5]:
# Check if the Pushover credentials are available in environment variables

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [6]:
# Function to send a push notification using Pushover

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [7]:
push("HEY!!") # Test the push function to ensure it works correctly

Push: HEY!!


In [8]:
# Example function to record user details and send a push notification

def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

In [9]:
# Example function to record an unknown question and send a push notification

def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [10]:
# Define the JSON schema for the record_user_details function

record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [11]:
# Define the JSON schema for the record_unknown_question function

record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [12]:
# Define the list of tools with their corresponding JSON schemas

tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [13]:
tools   # Print the tools to verify their structure

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name, if they provided it"},
     'notes': {'type': 'string',
      'description': "Any additional information about the conversation that's worth recording to give context"}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['quest

In [14]:
# This function can take a list of tool calls, and run them. This is the IF statement!!

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        # THE BIG IF STATEMENT!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)
        else:
            result = {}

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [15]:
# Test the handle_tool_calls function with a sample tool call for record_unknown_question

globals()["record_unknown_question"]("this is a really hard question")

Push: Recording this is a really hard question asked that I couldn't answer


{'recorded': 'ok'}

In [16]:
# Alternatively, we could use a more dynamic approach to call the tools without 
# hardcoding each one in an IF statement. This would involve using the globals() 
# function to get the tool function by name and then calling it with the arguments. 
# Here's how that would look:

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [17]:
# Pdf reading and extracting text from the LinkedIn profile

reader = PdfReader("linkedin/Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("linkedin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Shakiru Sikiru"

In [18]:
# Constructing the system prompt with the summary and LinkedIn profile information

system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [19]:
# This is the main chat function that takes in a user message and the conversation 
# history, and generates a response using the LLM. It also handles tool calls if 
# the LLM decides to call any tools.

def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:

        # Generate a response from the LLM, providing the tools as an option for it to call if needed

        response = openai.chat.completions.create(model="openai/gpt-oss-120b", messages=messages, tools=tools)

        finish_reason = response.choices[0].finish_reason
        
        # Check if the LLM called any tools. If it did, handle those tool calls 
        # and then continue the conversation loop to generate a new response with 
        # the tool results included in the messages. If it didn't call any tools, 
        # we are done and can return the final response content.
         
        if finish_reason=="tool_calls":
            assistant_message = response.choices[0].message
            tool_calls = assistant_message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(assistant_message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [ ]:
# Warnings can be ignored/suppressed for cleaner output in the Gradio interface

import warnings
warnings.filterwarnings("ignore", message=".*HTTP_422_UNPROCESSABLE_ENTITY.*")

# Launch the Gradio chat interface with the chat function as the backend
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Tool called: record_unknown_question
Push: Recording Do you have a patent? asked that I couldn't answer
Tool called: record_user_details
Push: Recording interest from Name not provided with email Shakiru@proforceintelligence.com and notes User provided email for follow‑up after asking about patents.
Tool called: record_unknown_question
Push: Recording Who is your favourite musician? asked that I couldn't answer
Tool called: record_user_details
Push: Recording interest from Name not provided with email Shakiru@proforceintelligence.com and notes User wants to get in touch after providing email.


## And now for deployment

This code is in `app.py`

We will deploy to HuggingFace Spaces.

Before you start: remember to update the files in the "me" directory - your LinkedIn profile and summary.txt - so that it talks about you! Also change `self.name = "Ed Donner"` in `app.py`..  

Also check that there's no README file within the 1_foundations directory. If there is one, please delete it. The deploy process creates a new README file in this directory for you.

And one more thing: this is optional, but you might want to delete the entire folder "community_contributions" within 1_foundations. You can always pull it from github again in the future. But if you don't, then this entire folder gets uploaded to HuggingFace even though we don't need it, and it's become quite large.

1. Visit https://huggingface.co and set up an account  
2. From the Avatar menu on the top right, choose Access Tokens. Choose "Create New Token". Give it WRITE permissions - it needs to have WRITE permissions! Keep a record of your new key.  
3. In the Terminal, run: `uv tool install 'huggingface_hub[cli]'` to install the HuggingFace tool, then `hf auth login --token YOUR_TOKEN_HERE`, like `hf auth login --token hf_xxxxxx`, to login at the command line with your key. Afterwards, run `hf auth whoami` to check you're logged in  
4. Take your new token and add it to your .env file: `HF_TOKEN=hf_xxx` for the future
5. From the 1_foundations folder, enter: `uv run gradio deploy` 
6. Follow its instructions: name it "career_conversation", specify app.py, choose cpu-basic as the hardware, say Yes to needing to supply secrets, provide your openai api key, your pushover user and token, and say "no" to github actions.  

Thank you Robert, James, Martins, Andras and Priya for these tips.  
Please read the next 2 sections - how to change your Secrets, and how to redeploy your Space (you may need to delete the README.md that gets created in this 1_foundations directory).

#### More about these secrets:

If you're confused by what's going on with these secrets: it just wants you to enter the key name and value for each of your secrets -- so you would enter:  
`OPENAI_API_KEY`  
Followed by:  
`sk-proj-...`  

And if you don't want to set secrets this way, or something goes wrong with it, it's no problem - you can change your secrets later:  
1. Log in to HuggingFace website  
2. Go to your profile screen via the Avatar menu on the top right  
3. Select the Space you deployed  
4. Click on the Settings wheel on the top right  
5. You can scroll down to change your secrets (Variables and Secrets section), delete the space, etc.

#### And now you should be deployed!

If you want to completely replace everything and start again with your keys, you may need to delete the README.md that got created in this 1_foundations folder.

Here is mine: https://huggingface.co/spaces/ed-donner/Career_Conversation

I just got a push notification that a student asked me how they can become President of their country 😂😂

For more information on deployment:

https://www.gradio.app/guides/sharing-your-app#hosting-on-hf-spaces

To delete your Space in the future:  
1. Log in to HuggingFace
2. From the Avatar menu, select your profile
3. Click on the Space itself and select the settings wheel on the top right
4. Scroll to the Delete section at the bottom
5. ALSO: delete the README file that Gradio may have created inside this 1_foundations folder (otherwise it won't ask you the questions the next time you do a gradio deploy)
